In [1]:
import pandas as pd

In [32]:
data = pd.read_csv('data/dataset komentar labelling.csv')
df = pd.DataFrame(data=data)
df

,Tanggal format,Komentar,Label IndoBERT
0,2026-06-30,oalah bentar juga bubar srimulat x kud aja bub...,Negatif
1,2026-06-30,Kopdes merah putih itu akan menghabisi pedagan...,Negatif
2,2026-06-30,Tidak akan bertahan lama bangkrut besar pasak ...,Negatif
3,2026-06-30,Kami yakin jadi bangunan mangkrak tiap kelurah...,Negatif
4,2026-06-29,ABS😂😂😂😂,Netral
...,...,...,...
2220,2026-06-19,Per unit Kdmp Nilai kontrak fisik 1.6 milyar D...,Positif
2221,2026-06-19,Lht saja 5 thn lg KDMP masih ada atau sdh jd F...,Positif
2222,2026-06-19,Wah asik nih Tumben ada ad 2 narsum,Netral
2223,2026-06-19,Jangan sampai KDKMP menjadi bancakan seperti BGN,Negatif


In [33]:
kamus_label = {
    "Positif": 1,
    "Netral": 0,
    "Negatif" : -1
}

df['Skor Sentimen'] = df['Label IndoBERT'].map(kamus_label)
df

,Tanggal format,Komentar,Label IndoBERT,Skor Sentimen
0,2026-06-30,oalah bentar juga bubar srimulat x kud aja bub...,Negatif,-1
1,2026-06-30,Kopdes merah putih itu akan menghabisi pedagan...,Negatif,-1
2,2026-06-30,Tidak akan bertahan lama bangkrut besar pasak ...,Negatif,-1
3,2026-06-30,Kami yakin jadi bangunan mangkrak tiap kelurah...,Negatif,-1
4,2026-06-29,ABS😂😂😂😂,Netral,0
...,...,...,...,...
2220,2026-06-19,Per unit Kdmp Nilai kontrak fisik 1.6 milyar D...,Positif,1
2221,2026-06-19,Lht saja 5 thn lg KDMP masih ada atau sdh jd F...,Positif,1
2222,2026-06-19,Wah asik nih Tumben ada ad 2 narsum,Netral,0
2223,2026-06-19,Jangan sampai KDKMP menjadi bancakan seperti BGN,Negatif,-1


**Feature Engginering**

x1 : rata rata sentimen harian

In [37]:
df_harian = pd.DataFrame()

df_harian['sentimen_avg'] = df.groupby('Tanggal format')['Skor Sentimen'].mean()

print(df_harian.head())

                sentimen_avg
Tanggal format              
2026-04-17         -0.450000
2026-04-18         -0.866667
2026-04-19         -0.560000
2026-04-20         -0.557895
2026-04-21         -0.656250


x2 : volume komentar harian

In [38]:
df_harian['volume_komentar'] = df.groupby('Tanggal format')['Komentar'].count()
print(df_harian.head())


                sentimen_avg  volume_komentar
Tanggal format                               
2026-04-17         -0.450000               20
2026-04-18         -0.866667               15
2026-04-19         -0.560000               50
2026-04-20         -0.557895               95
2026-04-21         -0.656250               64


x3 : rasio negatif harian

In [39]:
jumlah_negatif = (df['Skor Sentimen'] == -1).groupby(df['Tanggal format']).sum()

df_harian['rasio_negatif'] = jumlah_negatif / df_harian['volume_komentar']

print(df_harian.head())

                sentimen_avg  volume_komentar  rasio_negatif
Tanggal format                                              
2026-04-17         -0.450000               20       0.600000
2026-04-18         -0.866667               15       0.933333
2026-04-19         -0.560000               50       0.720000
2026-04-20         -0.557895               95       0.715789
2026-04-21         -0.656250               64       0.781250


x4 : volatilitas sentimen harian

In [40]:
df_harian['volatilitas_sentimen'] = df_harian['sentimen_avg'].rolling(3).std()

print(df_harian.head())

                sentimen_avg  volume_komentar  rasio_negatif  \
Tanggal format                                                 
2026-04-17         -0.450000               20       0.600000   
2026-04-18         -0.866667               15       0.933333   
2026-04-19         -0.560000               50       0.720000   
2026-04-20         -0.557895               95       0.715789   
2026-04-21         -0.656250               64       0.781250   

                volatilitas_sentimen  
Tanggal format                        
2026-04-17                       NaN  
2026-04-18                       NaN  
2026-04-19                  0.215930  
2026-04-20                  0.177665  
2026-04-21                  0.056188  


x5 : lag sentimen

In [41]:
df_harian['lag_sentimen'] = df_harian['sentimen_avg'].shift(1)

print(df_harian.head())

                sentimen_avg  volume_komentar  rasio_negatif  \
Tanggal format                                                 
2026-04-17         -0.450000               20       0.600000   
2026-04-18         -0.866667               15       0.933333   
2026-04-19         -0.560000               50       0.720000   
2026-04-20         -0.557895               95       0.715789   
2026-04-21         -0.656250               64       0.781250   

                volatilitas_sentimen  lag_sentimen  
Tanggal format                                      
2026-04-17                       NaN           NaN  
2026-04-18                       NaN     -0.450000  
2026-04-19                  0.215930     -0.866667  
2026-04-20                  0.177665     -0.560000  
2026-04-21                  0.056188     -0.557895  


**Menggabungkan dengan data IHSG**

In [42]:
data_ihsg = pd.read_csv('data/dataset IHSG (labeling).csv')

df_ihsg = pd.DataFrame(data_ihsg)
df_ihsg

,Tanggal_format,IHSG Close,Label
0,2026-06-19,6177.139160,1
1,2026-06-22,6116.689941,0
2,2026-06-23,6101.333008,0
3,2026-06-24,5883.880859,0
4,2026-06-25,5999.038086,1
5,2026-06-26,5896.133789,0
6,2026-06-29,5820.790039,0


In [44]:
df_gabungan = pd.merge(df_harian, df_ihsg, left_index=True, right_on='Tanggal_format', how='inner')

df_gabungan

,sentimen_avg,volume_komentar,rasio_negatif,volatilitas_sentimen,lag_sentimen,Tanggal_format,IHSG Close,Label
0,-0.396825,252,0.507937,0.170108,-0.449612,2026-06-19,6177.139160,1
1,-0.517857,56,0.678571,0.150501,-0.406977,2026-06-22,6116.689941,0
2,-0.545455,22,0.727273,0.073294,-0.517857,2026-06-23,6101.333008,0
3,-0.304348,23,0.608696,0.131960,-0.545455,2026-06-24,5883.880859,0
4,-1.000000,10,1.000000,0.353241,-0.304348,2026-06-25,5999.038086,1
5,-0.428571,7,0.571429,0.371011,-1.000000,2026-06-26,5896.133789,0
6,-0.400000,10,0.600000,0.303342,-1.000000,2026-06-29,5820.790039,0


In [45]:
print(df_gabungan.isna().sum())

sentimen_avg            0
volume_komentar         0
rasio_negatif           0
volatilitas_sentimen    0
lag_sentimen            0
Tanggal_format          0
IHSG Close              0
Label                   0
dtype: int64


In [46]:
df_gabungan[['sentimen_avg', 'volume_komentar', 'rasio_negatif', 'volatilitas_sentimen', 'lag_sentimen', 'Label']].to_csv('data/dataset modeling.csv', index=False, encoding='utf-8')
print('data disismpan')

data disismpan
